# 🎨 Proyecto Final Guiado: Análisis Perceptual y Limpieza de Datos en el Espacio de Color
### Introducción a Python para Ciencia de Datos e Inteligencia Artificial
---

## 🎯 Contexto del Proyecto

En este proyecto final trabajarás con los datos reales recopilados a través de la plataforma web del juego de mesa **Hues and Cues**, publicado en la investigación científica sobre percepción cromática humana e Inteligencia Artificial (*Alabau-Bosque et al.*).

En el juego, los participantes ven una palabra (por ejemplo: `"MANZANA"`, `"FLAMENCO"`, `"ESPERANZA"`, `"TRACTOR"`) y deben seleccionar una casilla del tablero (una cuadrícula de 480 tonos de color con coordenadas de la `A1` a la `P30`) que mejor represente el concepto visual de dicha palabra.

### 🎯 Objetivos de Aprendizaje:
1. **Manejo de Colecciones y Mapeos:** Uso intensivo de **diccionarios** (`dict`) para traducción de categorías y mapeo de casillas del tablero a nombres cromáticos (`.map()`).
2. **Creación de Funciones Modulares:** Definición de funciones reutilizables con parámetros, retorno y documentación (`def`, `return`, *docstrings*).
3. **Análisis Demográfico y Limpieza de Datos con Pandas:** Exploración de datos, cálculo de frecuencias relativas y filtrado estricto de participantes con **Daltonismo (Deficiencia de Visión Cromática - CVD)** siguiendo la metodología de calidad del artículo científico (Secciones 2.2 y 2.3 de `HuesAndCues.pdf`).
4. **Análisis de Percepción e Hipótesis Culturales:** Estudio comparativo de discrepancias cromáticas en personas daltónicas frente a tricrómatas, y evaluación del fenómeno de las **"anclas culturales"** (*TRACTOR* y *ESPERANZA*).
5. **Visualización de Datos con Seaborn y Matplotlib:** Creación de dashboards demográficos multivariable (`plt.subplots`), gráficos categóricos de barras (`sns.countplot`, `sns.barplot`) y gráficos de consenso ordenados.

## ⚙️ Fase 0: Configuración del Entorno y Librerías

Importaremos las librerías fundamentales de análisis de datos y visualización:
- `pandas` (`pd`): Manipulación tabular y operaciones sobre DataFrames.
- `numpy` (`np`): Soporte numérico y operaciones vectorizadas.
- `matplotlib.pyplot` (`plt`) y `seaborn` (`sns`): Creación de gráficos estadísticos y estéticos.

In [ ]:
# Importación de librerías esenciales
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración estética global
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("✅ Librerías importadas y entorno configurado correctamente.")

---
## 📥 Fase 1: Ingesta, Exploración y Diccionarios (EDA)

### 1.1 Carga de los Archivos de Datos
Debes cargar dos archivos fundamentales ubicados en el directorio del proyecto:
1. `HC.csv`: Contiene las 3.899 respuestas de los participantes humanos. Cárgalo en un DataFrame llamado `df_humanos`.
2. `board_color_categories.txt`: Mapea cada una de las 480 casillas del tablero a su color semántico básico (`brown`, `red`, `orange`, `yellow`, `green`, `blue`, `purple`, `pink`). Cárgalo en un DataFrame llamado `df_tablero`.

> 💡 **Pistas:**
> - El archivo `board_color_categories.txt` está separado por comas `,` y **no contiene cabecera**. Utiliza `header=None` y asigna nombres a las columnas: `names=['coordinate', 'color']`.
> - Revisa las dimensiones (`.shape`) y las primeras filas (`.head()`) de cada DataFrame.

In [ ]:
# TODO 1.1: Cargar los dos archivos de datos

# Tu código aquí:


### 1.2 Inspección de Tipos de Datos y Valores Faltantes
Comprueba si existen valores nulos en `df_humanos` con `.isnull().sum()` y muestra el resumen de columnas y tipos de datos con `.info()`.

In [ ]:
# TODO 1.2: Inspeccionar tipos de datos y valores nulos en df_humanos

# Tu código aquí:


### 1.3 Normalización de Categorías Semánticas con Diccionarios
Observa los valores únicos en `df_humanos['clueCategory']`. En el dataset original conviven etiquetas en inglés (`FOOD`, `STONE`, `ENVIRONMENT`, `SUBJETIVE`, `PLANT`) y en español (`COMESTIBLE`, `PIEDRA`, `ENTORNO`, `SUBJETIVO`, `PLANTA`).

1. Define un diccionario en Python llamado `traduccion_cat` que traduzca las etiquetas en inglés a su equivalente estándar en español:
   - `'FOOD'` $\rightarrow$ `'COMESTIBLE'`
   - `'STONE'` $\rightarrow$ `'PIEDRA'`
   - `'ENVIRONMENT'` $\rightarrow$ `'ENTORNO'`
   - `'SUBJETIVE'` $\rightarrow$ `'SUBJETIVO'`
   - `'PLANT'` $\rightarrow$ `'PLANTA'`
2. Aplica este diccionario a la columna `df_humanos['clueCategory']` usando el método `.replace(traduccion_cat)`.
3. Muestra el recuento final de cada categoría con `.value_counts()`.

In [ ]:
# TODO 1.3: Crear el diccionario de traducción y normalizar df_humanos['clueCategory']

# Tu código aquí:


### 1.4 Mapeo de Casilla a Color Semántico mediante Diccionarios
Para poder analizar qué color eligió cada participante (por ejemplo, si eligió un tono `pink`, `green`, `blue`, etc.), debemos enriquecer `df_humanos` con la información del tablero.

1. Crea un diccionario rápido `coord_a_color` a partir de `df_tablero` combinando las columnas `'coordinate'` y `'color'` usando `dict(zip(...))`.
2. Crea una nueva columna en `df_humanos` llamada `'color'` mapeando la columna `'coordinate'` con el diccionario creado (`df_humanos['coordinate'].map(coord_a_color)`).
3. Muestra las primeras 5 filas con las columnas `['clueId', 'word', 'clueCategory', 'coordinate', 'color']`.

In [ ]:
# TODO 1.4: Crear el diccionario coord_a_color y enriquecer df_humanos

# Tu código aquí:


---
## 👥 Fase 2: Perfil Demográfico y Filtrado de Participantes
> *Basado en las Secciones 2.2 y 2.3 del artículo científico (HuesAndCues.pdf)*

### 2.1 Análisis Demográfico de la Cohorte de Participantes
En `df_humanos`, cada fila representa la respuesta a una palabra individual. Sin embargo, cada participante respondió a una serie de 15 palabras.

1. Extrae los participantes únicos eliminando duplicados por la columna `'userId'` con `.drop_duplicates(subset=['userId'])` y guarda el resultado en un nuevo DataFrame llamado `df_usuarios`.
2. ¿Cuántos participantes únicos participaron en el experimento? (Usa `len(df_usuarios)` o `df_humanos['userId'].nunique()`).
3. Muestra la distribución de los participantes por:
   - **Género** (`df_usuarios['gender'].value_counts()`)
   - **Rango de Edad** (`df_usuarios['ageRange'].value_counts()`)
   - **Idioma** (`df_usuarios['language'].value_counts()`)
   - **Condición Visual / Daltonismo** (`df_usuarios['colorblind'].value_counts()` y en porcentaje con `normalize=True`)

In [ ]:
# TODO 2.1: Crear df_usuarios y calcular los descriptores demográficos

# Tu código aquí:


### 2.2 Filtrado de Calidad: Exclusión de Participantes con Daltonismo (CVD)
Tal y como explica el artículo en la **Sección 2.3**:
> *"To guarantee the integrity of the perceptual ground truth, we applied a strict filtering criterion based on self-reported visual capabilities, excluding participants who reported Color Vision Deficiency (CVD)..."*

1. Define una función llamada `filtrar_participantes(df, condicion_daltonismo='no')` con su correspondiente *docstring* que reciba un DataFrame y filtre las filas según la columna `'colorblind'`.
2. Utiliza tu función para crear dos DataFrames:
   - `df_tricromatas`: Participantes con visión normal (`colorblind == 'no'`). Esta será nuestra **verdad de referencia (ground truth)** para el resto del estudio.
   - `df_daltonicos`: Participantes que declararon daltonismo (`colorblind == 'YES'`).
3. Imprime cuántas respuestas totales contiene cada grupo y cuántos usuarios únicos representa cada uno.

In [ ]:
# TODO 2.2: Implementar función de filtrado y segmentar df_humanos
def filtrar_participantes(df, condicion_daltonismo='no'):
    """
    Filtra el DataFrame según la condición visual declarada en la columna 'colorblind'.
    
    Parámetros:
        df (pd.DataFrame): DataFrame con la columna 'colorblind'.
        condicion_daltonismo (str): 'no' para tricrómatas o 'YES' para daltónicos.
        
    Retorna:
        pd.DataFrame: DataFrame filtrado.
    """
    # Tu código aquí:
    pass

# Crear df_tricromatas y df_daltonicos


---
## 👁️ Fase 3: Estudio del Impacto del Daltonismo en la Percepción Cromática
> *Basado en la Sección 2.3 y Figura 4 del artículo científico*

### 3.1 Función de Comparación de Percepción Cromática por Palabra
Vamos a crear una función que nos permita contrastar qué color perciben los observadores con visión tricromática normal frente a los observadores con daltonismo para cualquier palabra.

1. Define una función `comparar_percepcion_palabra(palabra, df_tri, df_dal)` que:
   - Extraiga las respuestas de `df_tri` y de `df_dal` para la palabra indicada.
   - Calcule el color más votado (*moda*) en el grupo de tricrómatas y en el grupo de daltónicos.
   - Devuelva un diccionario o imprima un resumen con:
     - `palabra`
     - `color_tricromatas` (color mayoritario y porcentaje de consenso)
     - `color_daltonicos` (colores elegidos por los daltónicos)
     - `coincidencia` (`True` si el color más votado coincide en ambos grupos, `False` en caso contrario).

In [ ]:
# TODO 3.1: Definir la función comparar_percepcion_palabra
def comparar_percepcion_palabra(palabra, df_tri, df_dal):
    """
    Compara las elecciones de color entre observadores tricrómatas y daltónicos para una palabra.
    """
    # Tu código aquí:
    pass

# Prueba la función con 'CIELO' y con 'ZAFIRO'


### 3.2 Detección de Discrepancias y Confusión Espectral
El daltonismo suele afectar de manera muy pronunciada a la discriminación de tonos en el espectro **rojo-verde** o en tonos saturados con matices marrones/azules.

1. Itera sobre una lista de palabras presentes en el grupo de daltónicos (por ejemplo: `['ZAFIRO', 'JADE', 'MALVA', 'BLAS (Y EPI)', 'HURACÁN', 'CIELO', 'BARBIE', 'MENTA']`).
2. Ejecuta `comparar_percepcion_palabra()` para cada una y muestra los resultados.
3. Construye una tabla cruzada (`pd.crosstab`) con las elecciones de color de los daltónicos para observar qué colores eligen con mayor frecuencia.

In [ ]:
# TODO 3.2: Comparar la lista de palabras y construir la tabla de discrepancias
palabras_estudio = ['ZAFIRO', 'JADE', 'MALVA', 'BLAS (Y EPI)', 'HURACÁN', 'CIELO', 'BARBIE', 'MENTA']

# Tu código aquí:


---
## 🏛️ Fase 4: Anclas Culturales y Consenso Semántico
> *Basado en la Sección 2.2 y Figura 6 del artículo científico*

### 4.1 Análisis de Anclas Culturales (*TRACTOR* y *ESPERANZA*)
En el estudio se introdujeron palabras diseñadas como **"anclas culturales"** para comprobar si la memoria visual responde a la realidad física de los objetos o al imaginario cultural y lingüístico de la población:

1. **El caso de `TRACTOR`:** En el mundo físico agrícola, los tractores suelen ser de color **verde** (ej. John Deere) o **rojo** (ej. Massey Ferguson). Sin embargo, en la cultura popular hispanohablante existe una canción icónica (*"Mi tractor amarillo"*).
   - Filtra `df_tricromatas` para la palabra `'TRACTOR'`.
   - Calcula la distribución porcentual de los colores elegidos (`.value_counts(normalize=True) * 100`).
   - ¿Qué porcentaje eligió amarillo frente a verde o rojo?

2. **El caso de `ESPERANZA`:** Las emociones y conceptos abstractos no tienen un color físico real. Sin embargo, en español existe la metáfora arraigada del *"verde esperanza"*.
   - Filtra `df_tricromatas` para la palabra `'ESPERANZA'`.
   - Calcula la distribución porcentual de los colores elegidos.

In [ ]:
# TODO 4.1: Analizar las anclas culturales TRACTOR y ESPERANZA

# Tu código aquí:


### 4.2 Consenso Semántico por Categoría
¿Qué categorías de conceptos generan mayor acuerdo entre los humanos (consenso alto) y cuáles presentan mayor dispersión o subjetividad (consenso bajo)?

1. Define una función `calcular_consenso(serie_colores)` que reciba una serie de colores elegidos para una palabra y calcule el **porcentaje de votos que obtuvo el color mayoritario**:
   $$\text{Consenso (\%)} = \frac{\text{votos del color más repetido}}{\text{total de votos de la palabra}} \times 100$$
2. Agrupa `df_tricromatas` por `['clueCategory', 'word']` y calcula el consenso para cada palabra aplicando tu función con `.apply()` o agregaciones.
3. Agrupa por `'clueCategory'` para obtener el **consenso medio de cada categoría** y ordénalo de mayor a menor con `.sort_values(ascending=False)`.

In [ ]:
# TODO 4.2: Función calcular_consenso y agregación por categoría semántica
def calcular_consenso(serie_colores):
    """
    Calcula el porcentaje de acuerdo en torno al color modal (más votado).
    """
    # Tu código aquí:
    pass

# Calcular consenso por palabra y promedio por categoría


---
## 📊 Fase 5: Visualización de Datos (DataViz con Matplotlib y Seaborn)

### 5.1 Gráfico 1: Dashboard Demográfico del Estudio (4 Subplots con `plt.subplots`)
Crea una figura con $2 \times 2$ subplots (`fig, axes = plt.subplots(2, 2, figsize=(15, 10))`) utilizando `df_usuarios`:
- **Subplot (0, 0):** Distribución por Rango de Edad (`sns.countplot` en `x='ageRange'`).
- **Subplot (0, 1):** Distribución por Género (`sns.countplot` en `x='gender'`).
- **Subplot (1, 0):** Distribución por Idioma (`sns.countplot` en `x='language'`).
- **Subplot (1, 1):** Condición Visual: Daltónicos vs. Tricrómatas (`sns.countplot` en `x='colorblind'`).

Personaliza títulos, etiquetas de ejes y utiliza `plt.tight_layout()`.

In [ ]:
# TODO 5.1: Crear el Dashboard Demográfico (2x2 subplots)

# Tu código aquí:


### 5.2 Gráfico 2: Comparativa Visual: Percepción de Tricrómatas vs. Daltónicos
Elige dos palabras con discrepancia evidente (por ejemplo, `'ZAFIRO'` y `'JADE'`).

Crea una figura con 2 subplots (`1x2`):
- **Panel Izquierdo (`ZAFIRO`):** Gráfico de barras (`sns.countplot` o `.value_counts().plot(kind='bar')`) comparando los colores elegidos por tricrómatas frente a daltónicos.
- **Panel Derecho (`JADE`):** Gráfico de barras análogo para la palabra `JADE`.

In [ ]:
# TODO 5.2: Gráfico comparativo de percepción cromática en palabras críticas

# Tu código aquí:


### 5.3 Gráfico 3: Visualización del Fenómeno de las Anclas Culturales
Crea una figura con 2 subplots (`1x2`):
- **Panel Izquierdo:** Gráfico de barras con la distribución de colores para `'TRACTOR'` en `df_tricromatas`, destacando el color dominante (`yellow`).
- **Panel Derecho:** Gráfico de barras con la distribución de colores para `'ESPERANZA'`, destacando el color dominante (`green`).

In [ ]:
# TODO 5.3: Visualización de las Anclas Culturales (TRACTOR y ESPERANZA)

# Tu código aquí:


### 5.4 Gráfico 4: Ranking de Consenso Semántico por Categoría
Crea un gráfico de barras horizontal (`sns.barplot` con orientación horizontal `x='consenso', y='clueCategory'`) que muestre el consenso medio de color para cada una de las categorías semánticas, ordenado de mayor a menor acuerdo.

In [ ]:
# TODO 5.4: Gráfico de barras horizontal del consenso semántico por categoría

# Tu código aquí:


---
## 💡 Fase 6: Extracción de Conclusiones y Discusión Crítica

> ✍️ **Preguntas de Reflexión para el Alumno:**
>
> 1. **Calidad de Datos y Validez Perceptual:** ¿Por qué en estudios psicofísicos y de percepción visual es imprescindible filtrar y aislar a los participantes con deficiencias visuales (como el daltonismo) antes de calcular la verdad de referencia (*ground truth*)?
>
> 2. **Patrón de Confusión Cromática:** Observando las palabras analizadas (*ZAFIRO*, *JADE*, *BLAS*), ¿qué tipo de confusión cromática se manifiesta en las personas daltónicas? ¿En qué casos coinciden plenamente con los tricrómatas?
>
> 3. **Influencia del Lenguaje y la Cultura:** ¿Qué nos demuestra el caso de *TRACTOR* y *ESPERANZA* sobre la cognición humana? ¿La memoria del color es puramente física o está modulada por la cultura popular y el lenguaje materno?
>
> 4. **Consenso entre Categorías:** ¿Por qué categorías como *COMESTIBLE* o *ANIMAL* tienen mayor consenso que *SUBJETIVO* o *PIEDRA*?